# Cohorts and retention

Practice notebook for the datafromzero SQL lesson **Cohorts and retention**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Assign each customer to a cohort, then size the cohorts

In [ ]:
con.sql("""
SELECT customer_id, signup_date,
       CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month
FROM customers
WHERE signup_date >= DATE '2026-01-01'
ORDER BY customer_id
LIMIT 5
""").show()

In [ ]:
con.sql("""
SELECT CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month, COUNT(*) AS cohort_size
FROM customers
WHERE signup_date >= DATE '2026-01-01'
GROUP BY cohort_month
ORDER BY cohort_month
""").show()

### Step 2. List who was active in which month

In [ ]:
con.sql("""
WITH cohorts AS (
  SELECT customer_id, CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month
  FROM customers
  WHERE signup_date >= DATE '2026-01-01'
),
activity AS (
  SELECT DISTINCT c.cohort_month, c.customer_id,
         DATE_DIFF('month', c.cohort_month, CAST(o.ordered_at AS DATE)) AS months_since
  FROM cohorts c
  JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
)
SELECT * FROM activity
ORDER BY customer_id, months_since
LIMIT 6
""").show()

### Step 3. Count active customers per cell and divide by the cohort size

In [ ]:
con.sql("""
WITH cohorts AS (
  SELECT customer_id, CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month
  FROM customers
  WHERE signup_date >= DATE '2026-01-01'
),
cohort_sizes AS (
  SELECT cohort_month, COUNT(*) AS cohort_size
  FROM cohorts
  GROUP BY cohort_month
),
activity AS (
  SELECT DISTINCT c.cohort_month, c.customer_id,
         DATE_DIFF('month', c.cohort_month, CAST(o.ordered_at AS DATE)) AS months_since
  FROM cohorts c
  JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
),
retained AS (
  SELECT cohort_month, months_since, COUNT(*) AS customers
  FROM activity
  GROUP BY cohort_month, months_since
),
retention AS (
  SELECT r.cohort_month, r.months_since, s.cohort_size, r.customers,
         ROUND(100.0 * r.customers / s.cohort_size, 1) AS pct
  FROM retained r
  JOIN cohort_sizes s ON s.cohort_month = r.cohort_month
)
SELECT * FROM retention
ORDER BY cohort_month, months_since
""").show()

### Step 4. Turn it into the triangle

In [ ]:
con.sql("""
WITH cohorts AS (
  SELECT customer_id, CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month
  FROM customers
  WHERE signup_date >= DATE '2026-01-01'
),
cohort_sizes AS (
  SELECT cohort_month, COUNT(*) AS cohort_size
  FROM cohorts
  GROUP BY cohort_month
),
activity AS (
  SELECT DISTINCT c.cohort_month, c.customer_id,
         DATE_DIFF('month', c.cohort_month, CAST(o.ordered_at AS DATE)) AS months_since
  FROM cohorts c
  JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
),
retained AS (
  SELECT cohort_month, months_since, COUNT(*) AS customers
  FROM activity
  GROUP BY cohort_month, months_since
),
retention AS (
  SELECT r.cohort_month, r.months_since, s.cohort_size, r.customers,
         ROUND(100.0 * r.customers / s.cohort_size, 1) AS pct
  FROM retained r
  JOIN cohort_sizes s ON s.cohort_month = r.cohort_month
)
SELECT cohort_month,
       MAX(cohort_size) AS cohort_size,
       MAX(CASE WHEN months_since = 0 THEN pct END) AS m0,
       MAX(CASE WHEN months_since = 1 THEN pct END) AS m1,
       MAX(CASE WHEN months_since = 2 THEN pct END) AS m2,
       MAX(CASE WHEN months_since = 3 THEN pct END) AS m3,
       MAX(CASE WHEN months_since = 4 THEN pct END) AS m4,
       MAX(CASE WHEN months_since = 5 THEN pct END) AS m5
FROM retention
GROUP BY cohort_month
ORDER BY cohort_month
""").show()

### Step 5. Verify one cell by a second method

In [ ]:
con.sql("""
SELECT COUNT(*) AS cohort_size,
       COUNT(CASE WHEN april_orders > 0 THEN 1 END) AS active_in_april
FROM (
  SELECT c.customer_id,
         (SELECT COUNT(*) FROM orders o
          WHERE o.customer_id = c.customer_id AND o.status = 'completed'
            AND o.ordered_at >= DATE '2026-04-01' AND o.ordered_at < DATE '2026-05-01') AS april_orders
  FROM customers c
  WHERE c.signup_date >= DATE '2026-02-01' AND c.signup_date < DATE '2026-03-01'
) AS feb_customers
""").show()

## Exercise 1. The change from month to month, January cohort

The five steps of step 3 are written for you. Only the last `SELECT` is yours: fill the window. Expected: 6 rows, 4
columns (the month, the share, the change); the first change is `NULL`.

In [ ]:
con.sql("""
WITH cohorts AS (
  SELECT customer_id, CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month
  FROM customers
  WHERE signup_date >= DATE '2026-01-01'
),
cohort_sizes AS (
  SELECT cohort_month, COUNT(*) AS cohort_size
  FROM cohorts
  GROUP BY cohort_month
),
activity AS (
  SELECT DISTINCT c.cohort_month, c.customer_id,
         DATE_DIFF('month', c.cohort_month, CAST(o.ordered_at AS DATE)) AS months_since
  FROM cohorts c
  JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
),
retained AS (
  SELECT cohort_month, months_since, COUNT(*) AS customers
  FROM activity
  GROUP BY cohort_month, months_since
),
retention AS (
  SELECT r.cohort_month, r.months_since, s.cohort_size, r.customers,
         ROUND(100.0 * r.customers / s.cohort_size, 1) AS pct
  FROM retained r
  JOIN cohort_sizes s ON s.cohort_month = r.cohort_month
)
SELECT cohort_month, months_since, pct
       -- add the change from the previous month here, with LAG and a window
FROM retention
WHERE cohort_month = DATE '2026-01-01'
ORDER BY months_since
""").show()

## Exercise 2. First completed order by month

Before you run anything, write one sentence in the cell below: what you expect of January compared with the other
months. Then count the customers by the month of their first completed order, and count how many of January's customers
signed up before 2026. Leave out the walk-in order.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Query 1: customers by the month of their first completed order

In [ ]:
# Query 2: how many of January's customers signed up before 2026

## Exercise 3. Spot the flaw

A colleague drops the `DISTINCT` from the `activity` step. Run the query: January, month 1, reads `209.1`. Find out what
went wrong and write the fix.

In [ ]:
con.sql("""
WITH cohorts AS (
  SELECT customer_id, CAST(DATE_TRUNC('month', signup_date) AS DATE) AS cohort_month
  FROM customers
  WHERE signup_date >= DATE '2026-01-01'
),
cohort_sizes AS (
  SELECT cohort_month, COUNT(*) AS cohort_size
  FROM cohorts
  GROUP BY cohort_month
),
activity AS (
  SELECT c.cohort_month, c.customer_id,
         DATE_DIFF('month', c.cohort_month, CAST(o.ordered_at AS DATE)) AS months_since
  FROM cohorts c
  JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
),
retained AS (
  SELECT cohort_month, months_since, COUNT(*) AS customers
  FROM activity
  GROUP BY cohort_month, months_since
),
retention AS (
  SELECT r.cohort_month, r.months_since, s.cohort_size, r.customers,
         ROUND(100.0 * r.customers / s.cohort_size, 1) AS pct
  FROM retained r
  JOIN cohort_sizes s ON s.cohort_month = r.cohort_month
)
SELECT * FROM retention
WHERE cohort_month = DATE '2026-01-01'
ORDER BY months_since
""").show()

In [ ]:
# Your corrected query: